# 08 - Regime Analysis

**Purpose.** Post-hoc classify hours into low / medium / high volatility tertiles based on rolling sigma(r_aave) and run the buy-and-hold baseline per regime to surface conditional performance (PROJECT_2_PLAN.md S6.4 + S9.5).

**Prerequisites.** Synthetic-or-real joined panel.

**Expected runtime.** < 1 minute.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


## 1. Rolling volatility (24h sigma of Aave rate)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vol = df['r_aave'].rolling(24, min_periods=12).std()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(vol.index, vol.values * 1e4, color='C3')
ax.set_title('Rolling 24h std(r_aave) (1e-4 units)')
ax.set_xlabel('time'); ax.set_ylabel('sigma * 1e4')
plt.tight_layout(); plt.show()

# Caption: Volatility regimes are visible as sustained high-sigma
# bands. The plan splits these into tertiles for conditional metrics.


## 2. Tertile-split

In [ ]:
vol_q1 = vol.quantile(1/3)
vol_q2 = vol.quantile(2/3)
df_reg = df.assign(vol=vol).dropna()
df_reg['regime'] = np.where(df_reg['vol'] <= vol_q1, 'low',
                  np.where(df_reg['vol'] <= vol_q2, 'med', 'high'))
print(df_reg['regime'].value_counts())


## 3. Run baseline per regime (buy-and-hold)

In [ ]:
# Demo: report average rate per regime as a stand-in for full backtest.
# A full per-regime backtest splits observations into regime buckets and
# runs BuyAndHoldAaveStrategy on each; this requires the observations
# builder to accept a row mask. Stubbed here for clarity.
summary = df_reg.groupby('regime')[['r_aave', 'r_compound']].agg(['mean', 'std'])
summary


## 4. Compare metrics across regimes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = ['low', 'med', 'high']
means = [df_reg[df_reg['regime'] == r]['r_aave'].mean() * 100 for r in order]
ax.bar(order, means, color=['C0', 'C1', 'C3'])
ax.set_ylabel('mean Aave APY % per regime')
ax.set_title('Conditional rate level by volatility regime')
plt.tight_layout(); plt.show()


## 5. Skip gracefully when real data missing

In [ ]:
print(f'Used {"real" if is_real else "synthetic"} panel for this regime analysis.')
if not is_real:
    print('Note: synthetic data is locally smooth - regime structure is\n'
          '  not as informative as on the real 18-month panel. Re-run\n'
          '  once data/cached/joined_clean.parquet is materialised.')


## Next steps

- Wire `build_all(synthetic=...)` to accept a row mask so we can run the   full strategy suite per regime, not just compute mean rates.
- Add per-regime Sharpe / MaxDD tables to the whitepaper.

Relevant plan section: **PROJECT_2_PLAN.md S6.4 (Regime Split) and S9.5 (Regime-Conditional Metrics).**
